<a href="https://colab.research.google.com/github/mperumal-usd/capstone_team_3/blob/main/notebooks/COLAB_MERT_FAISS_Recall_Eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MERT Fine-tuned: FAISS Index + Recall@K Evaluation

Builds a FAISS index from classical WAV chunk embeddings produced by each fine-tuned MERT
checkpoint, then evaluates Recall@1/3/5/10 on external query WAV chunks.

## Sections
- **Section 0** — Setup (install deps, mount Drive, config)
- **Section 1** — Helper functions (model loading, embedding, FAISS)
- **Section 2** — Gallery embedding generation (resume-safe, per checkpoint)
- **Section 3** — FAISS index build (skips if index already on Drive)
- **Section 4** — Query embedding generation
- **Section 5** — Recall@1,3,5,10 computation & comparison table

**Resume after disconnect:** Run all cells top-to-bottom. Each section checks Drive
for its output and skips if already complete. Embedding generation saves a partial
`.npy` + progress JSON every `SAVE_EVERY` files so it can continue mid-run.

In [ ]:
# ── Section 0A: Install dependencies ─────────────────────────────────────────
!pip install -q transformers peft accelerate librosa numpy pandas tqdm
!pip install -q faiss-gpu 2>/dev/null || pip install -q faiss-cpu
print("Dependencies installed.")

In [ ]:
# ── Section 0B: Mount Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Section 0C: Config ────────────────────────────────────────────────────────
import os

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE_ROOT = "/content/drive/MyDrive/AAI-590 Capstone"

# List of fine-tuned checkpoint directories (each must have checkpoints/lora_best/ inside)
# Add or remove entries to match your actual Drive structure.
CHECKPOINT_CONFIGS = [
    {
        "name": "mert_v3",
        "base_dir":   os.path.join(DRIVE_ROOT, "checkpoints3"),
        "lora_dir":   os.path.join(DRIVE_ROOT, "checkpoints3", "checkpoints", "lora_best"),
        "cache_dir":  os.path.join(DRIVE_ROOT, "checkpoints3", "model_cache"),
    },
    {
        "name": "mert_v4",
        "base_dir":   os.path.join(DRIVE_ROOT, "checkpoints4"),
        "lora_dir":   os.path.join(DRIVE_ROOT, "checkpoints4", "checkpoints", "lora_best"),
        "cache_dir":  os.path.join(DRIVE_ROOT, "checkpoints4", "model_cache"),
    },
    # Add more checkpoints here, e.g.:
    # {
    #     "name": "mert_v5",
    #     "base_dir":  os.path.join(DRIVE_ROOT, "checkpoints5"),
    #     "lora_dir":  os.path.join(DRIVE_ROOT, "checkpoints5", "checkpoints", "lora_best"),
    #     "cache_dir": os.path.join(DRIVE_ROOT, "checkpoints5", "model_cache"),
    # },
]

# Gallery: classical WAV chunks used to build the FAISS index
# Structure: GALLERY_PATH/<composer>/<song>/<song>_chunk_N.wav
GALLERY_PATH = os.path.join(DRIVE_ROOT, "AAI590_FinalProject", "ChunkSamples")

# Query: external WAV chunks to evaluate Recall@K on
# Structure can be flat (all .wav in one folder) or same nested structure as GALLERY_PATH.
# If nested, composer labels are inferred from the folder name (like gallery).
# If flat, all files are treated as one class (composer="unknown") — override QUERY_FLAT_LABELS
# with a dict {filename: composer_label} for proper Recall@K evaluation.
QUERY_PATH = os.path.join(DRIVE_ROOT, "AAI590_FinalProject", "QueryChunks")

# Where to save embeddings, FAISS indexes, and results
EVAL_BASE = os.path.join(DRIVE_ROOT, "FAISS_Eval")

# Recall thresholds
RECALL_AT_K = [1, 3, 5, 10]

# Retrieval unit: "composer" (same composer = relevant) or "song" (same song = relevant)
# Use "composer" if query chunks are from different recordings of the same composer.
RETRIEVAL_UNIT = "composer"  # or "song"

# Audio config
SR = 24000

# Embedding batch size (reduce if OOM)
EMBED_BATCH_SIZE = 16

# Save partial embeddings every N files (for resume)
SAVE_EVERY = 200

# ── Runtime ───────────────────────────────────────────────────────────────────
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Create output dirs
for sub in ["embeddings", "indexes", "results"]:
    os.makedirs(os.path.join(EVAL_BASE, sub), exist_ok=True)
print(f"Output base: {EVAL_BASE}")

# Validate checkpoint dirs exist
for cfg in CHECKPOINT_CONFIGS:
    exists = os.path.isdir(cfg["lora_dir"])
    print(f"  {cfg['name']}: lora_dir {'OK' if exists else 'MISSING — check path'}")

## Section 1 — Helper Functions

In [ ]:
# ── Section 1: Helpers ────────────────────────────────────────────────────────
import json
import re
import numpy as np
import torch
import torch.nn.functional as F
import librosa
from tqdm import tqdm
from transformers import AutoModel, AutoProcessor
from peft import PeftModel
import faiss

MODEL_NAME = "m-a-p/MERT-v1-95M"


# ── Model loading ─────────────────────────────────────────────────────────────
def load_model(cfg):
    """Load base MERT + LoRA adapter for a given checkpoint config."""
    cache_dir = cfg["cache_dir"]
    if os.path.isdir(cache_dir) and os.listdir(cache_dir):
        print(f"[{cfg['name']}] Loading MERT from Drive cache: {cache_dir}")
        processor  = AutoProcessor.from_pretrained(cache_dir, trust_remote_code=True)
        base_model = AutoModel.from_pretrained(cache_dir, trust_remote_code=True)
    else:
        print(f"[{cfg['name']}] Downloading MERT from HuggingFace...")
        processor  = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
        base_model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True)
        os.makedirs(cache_dir, exist_ok=True)
        processor.save_pretrained(cache_dir)
        base_model.save_pretrained(cache_dir)
        print(f"[{cfg['name']}] Cached to: {cache_dir}")

    model = PeftModel.from_pretrained(base_model, cfg["lora_dir"])
    model.eval().to(DEVICE)
    print(f"[{cfg['name']}] LoRA adapter loaded from {cfg['lora_dir']}")
    return model, processor


# ── Embedding ─────────────────────────────────────────────────────────────────
def wav_to_embedding(audio_paths, processor, model, sr=SR, batch_size=EMBED_BATCH_SIZE):
    """Load WAV files and return L2-normalised 768-dim embeddings (CPU numpy)."""
    all_embs = []
    use_amp  = DEVICE.type == "cuda"

    for i in range(0, len(audio_paths), batch_size):
        batch_paths = audio_paths[i : i + batch_size]
        batch_audio = []
        for p in batch_paths:
            audio, _ = librosa.load(p, sr=sr, mono=True)
            batch_audio.append(audio)

        # Pad to same length
        max_len = max(a.shape[0] for a in batch_audio)
        padded  = [np.pad(a, (0, max_len - a.shape[0])) for a in batch_audio]

        inputs = processor(
            padded, sampling_rate=sr, return_tensors="pt", padding=True
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp):
            outputs = model(**inputs)
            emb = outputs.last_hidden_state.mean(dim=1)  # (B, 768)
            emb = F.normalize(emb, p=2, dim=1)

        all_embs.append(emb.cpu().float().numpy())

    return np.concatenate(all_embs, axis=0)  # (N, 768)


# ── File scanning ─────────────────────────────────────────────────────────────
def scan_wav_files(root_path):
    """
    Walk root_path and return:
      paths  : list of absolute WAV file paths
      composers: list of composer labels (top-level folder name)
      songs   : list of song labels (second-level folder name)

    Supports both nested (root/composer/song/*.wav) and
    flat (root/*.wav) layouts.
    """
    paths, composers, songs = [], [], []
    for dirpath, dirnames, filenames in os.walk(root_path):
        dirnames.sort()  # deterministic order
        for fname in sorted(filenames):
            if not fname.lower().endswith(".wav"):
                continue
            full = os.path.join(dirpath, fname)
            rel  = os.path.relpath(full, root_path)
            parts = rel.split(os.sep)
            if len(parts) >= 3:         # composer/song/file.wav
                composer = parts[0]
                song     = parts[1]
            elif len(parts) == 2:       # song/file.wav
                composer = parts[0]
                song     = parts[0]
            else:                       # flat layout
                composer = "unknown"
                song     = os.path.splitext(fname)[0]
            paths.append(full)
            composers.append(composer)
            songs.append(song)
    return paths, composers, songs


# ── FAISS helpers ─────────────────────────────────────────────────────────────
def build_faiss_index(embeddings):
    """Build an inner-product (cosine) FAISS index from L2-normalised embeddings."""
    dim   = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # exact search; embeddings are already L2-normalised
    if DEVICE.type == "cuda" and hasattr(faiss, 'StandardGpuResources'):
        res   = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index)
        print("FAISS: running on GPU")
    else:
        print("FAISS: running on CPU")
    index.add(embeddings.astype(np.float32))
    return index


def recall_at_k(query_labels, gallery_labels, scores, ks=RECALL_AT_K):
    """
    Compute Recall@K for a set of queries.

    Parameters
    ----------
    query_labels   : list of N labels (one per query)
    gallery_labels : list of M labels (one per gallery item)
    scores         : np.ndarray (N, max_K) — top-max_K gallery indices per query
                     (output of faiss.search)
    ks             : list of K values to evaluate

    Returns
    -------
    dict  {k: recall_float}
    """
    gallery_labels_arr = np.array(gallery_labels)
    results = {}
    max_k   = scores.shape[1]

    for k in ks:
        if k > max_k:
            print(f"Warning: requested K={k} > retrieved={max_k}; skipping")
            continue
        hits = 0
        for i, qlabel in enumerate(query_labels):
            top_k_labels = gallery_labels_arr[scores[i, :k]]
            if qlabel in top_k_labels:
                hits += 1
        results[k] = hits / len(query_labels)

    return results


print("Helper functions defined.")

## Section 2 — Gallery Embedding Generation

For each checkpoint, embed all WAV files in `GALLERY_PATH`.
Saves partial results every `SAVE_EVERY` files — safe to re-run after disconnect.

In [ ]:
# ── Section 2A: Scan gallery files (once) ────────────────────────────────────
gallery_paths, gallery_composers, gallery_songs = scan_wav_files(GALLERY_PATH)
print(f"Gallery files found: {len(gallery_paths)}")
print(f"Unique composers   : {len(set(gallery_composers))}")
print(f"Unique songs       : {len(set(gallery_songs))}")

# Save the manifest to Drive (used to reconstruct labels after resume)
import pandas as pd
gallery_manifest_path = os.path.join(EVAL_BASE, "gallery_manifest.csv")
gallery_df = pd.DataFrame({
    "path":     gallery_paths,
    "composer": gallery_composers,
    "song":     gallery_songs,
})
gallery_df.to_csv(gallery_manifest_path, index=False)
print(f"Manifest saved → {gallery_manifest_path}")

In [ ]:
# ── Section 2B: Generate embeddings (with resume) ─────────────────────────────
# For each checkpoint we produce: embeddings/<name>/gallery_embeddings.npy
# Progress tracked in: embeddings/<name>/progress.json

for cfg in CHECKPOINT_CONFIGS:
    name   = cfg["name"]
    emb_dir = os.path.join(EVAL_BASE, "embeddings", name)
    os.makedirs(emb_dir, exist_ok=True)

    final_emb_path  = os.path.join(emb_dir, "gallery_embeddings.npy")
    progress_path   = os.path.join(emb_dir, "progress.json")
    partial_emb_path = os.path.join(emb_dir, "gallery_embeddings_partial.npy")

    # Skip if already done
    if os.path.exists(final_emb_path):
        print(f"[{name}] Gallery embeddings already exist — skipping.")
        continue

    # Resume: load partial results
    start_idx      = 0
    partial_embeds = []
    if os.path.exists(progress_path):
        with open(progress_path) as f:
            progress = json.load(f)
        start_idx = progress["done"]
        if os.path.exists(partial_emb_path):
            partial_embeds = [np.load(partial_emb_path)]
            print(f"[{name}] Resuming from file {start_idx}/{len(gallery_paths)}")
        else:
            start_idx = 0

    print(f"[{name}] Loading model...")
    model, processor = load_model(cfg)

    remaining_paths = gallery_paths[start_idx:]
    chunk_embeds    = list(partial_embeds)
    processed       = start_idx

    pbar = tqdm(range(0, len(remaining_paths), EMBED_BATCH_SIZE),
                desc=f"[{name}] Embedding gallery")

    for batch_start in pbar:
        batch = remaining_paths[batch_start : batch_start + EMBED_BATCH_SIZE]
        embs  = wav_to_embedding(batch, processor, model)
        chunk_embeds.append(embs)
        processed += len(batch)
        pbar.set_postfix({"done": processed})

        # Save partial checkpoint
        if processed % SAVE_EVERY < EMBED_BATCH_SIZE or processed == len(gallery_paths):
            np.save(partial_emb_path, np.concatenate(chunk_embeds, axis=0))
            with open(progress_path, "w") as f:
                json.dump({"done": processed, "total": len(gallery_paths)}, f)

    # Finalize
    final_embeddings = np.concatenate(chunk_embeds, axis=0)
    np.save(final_emb_path, final_embeddings)
    print(f"[{name}] Saved {final_embeddings.shape} embeddings → {final_emb_path}")

    # Clean up partial files
    for p in [partial_emb_path, progress_path]:
        if os.path.exists(p):
            os.remove(p)

    # Free GPU memory before loading next model
    del model
    torch.cuda.empty_cache()

print("\nAll gallery embeddings complete.")

## Section 3 — Build FAISS Index

One index per checkpoint. Skips if index file already exists on Drive.

In [ ]:
# ── Section 3: Build FAISS indexes ────────────────────────────────────────────
faiss_indexes = {}   # name → faiss index (in-memory)

for cfg in CHECKPOINT_CONFIGS:
    name          = cfg["name"]
    emb_path      = os.path.join(EVAL_BASE, "embeddings", name, "gallery_embeddings.npy")
    index_path    = os.path.join(EVAL_BASE, "indexes", f"{name}_faiss.index")

    if not os.path.exists(emb_path):
        print(f"[{name}] Embeddings not found — run Section 2 first.")
        continue

    embeddings = np.load(emb_path).astype(np.float32)
    print(f"[{name}] Gallery embeddings: {embeddings.shape}")

    # Build or load CPU-serializable index
    if os.path.exists(index_path):
        print(f"[{name}] Loading existing FAISS index from Drive...")
        cpu_index = faiss.read_index(index_path)
    else:
        print(f"[{name}] Building FAISS IndexFlatIP...")
        dim       = embeddings.shape[1]
        cpu_index = faiss.IndexFlatIP(dim)
        cpu_index.add(embeddings)
        faiss.write_index(cpu_index, index_path)
        print(f"[{name}] Index saved → {index_path}  ({cpu_index.ntotal} vectors)")

    # Optionally move to GPU for faster search
    if DEVICE.type == "cuda" and hasattr(faiss, 'StandardGpuResources'):
        res = faiss.StandardGpuResources()
        gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
        faiss_indexes[name] = gpu_index
        print(f"[{name}] Index moved to GPU")
    else:
        faiss_indexes[name] = cpu_index

print("\nFAISS indexes ready:", list(faiss_indexes.keys()))

## Section 4 — Query Embedding Generation

Embeds the external query WAV chunks. Saved to Drive so re-runs are fast.

In [ ]:
# ── Section 4A: Scan query files ──────────────────────────────────────────────
if not os.path.isdir(QUERY_PATH):
    raise FileNotFoundError(
        f"QUERY_PATH not found: {QUERY_PATH}\n"
        "Update QUERY_PATH in Section 0C to point to your external query WAV chunks."
    )

query_paths, query_composers, query_songs = scan_wav_files(QUERY_PATH)
print(f"Query files found: {len(query_paths)}")
print(f"Unique composers : {len(set(query_composers))}")
print(f"Sample composers : {sorted(set(query_composers))[:10]}")

# Save query manifest
query_manifest_path = os.path.join(EVAL_BASE, "query_manifest.csv")
query_df = pd.DataFrame({
    "path":     query_paths,
    "composer": query_composers,
    "song":     query_songs,
})
query_df.to_csv(query_manifest_path, index=False)
print(f"Query manifest saved → {query_manifest_path}")

# Choose label column based on RETRIEVAL_UNIT
assert RETRIEVAL_UNIT in ("composer", "song"), "RETRIEVAL_UNIT must be 'composer' or 'song'"
query_labels   = query_composers   if RETRIEVAL_UNIT == "composer" else query_songs
gallery_labels = gallery_composers if RETRIEVAL_UNIT == "composer" else gallery_songs
print(f"Retrieval unit: {RETRIEVAL_UNIT}")

In [ ]:
# ── Section 4B: Embed queries (with resume) ───────────────────────────────────
query_emb_cache = {}   # name → np.ndarray (N_q, 768)

for cfg in CHECKPOINT_CONFIGS:
    name     = cfg["name"]
    emb_dir  = os.path.join(EVAL_BASE, "embeddings", name)
    os.makedirs(emb_dir, exist_ok=True)

    final_query_emb  = os.path.join(emb_dir, "query_embeddings.npy")
    partial_query_emb = os.path.join(emb_dir, "query_embeddings_partial.npy")
    progress_path    = os.path.join(emb_dir, "query_progress.json")

    if os.path.exists(final_query_emb):
        print(f"[{name}] Query embeddings already exist — loading.")
        query_emb_cache[name] = np.load(final_query_emb).astype(np.float32)
        continue

    # Resume
    start_idx      = 0
    partial_embeds = []
    if os.path.exists(progress_path):
        with open(progress_path) as f:
            progress = json.load(f)
        start_idx = progress["done"]
        if os.path.exists(partial_query_emb):
            partial_embeds = [np.load(partial_query_emb)]
            print(f"[{name}] Resuming query embedding from {start_idx}/{len(query_paths)}")

    print(f"[{name}] Loading model for query embedding...")
    model, processor = load_model(cfg)

    remaining = query_paths[start_idx:]
    chunk_embeds = list(partial_embeds)
    processed    = start_idx

    for batch_start in tqdm(range(0, len(remaining), EMBED_BATCH_SIZE),
                            desc=f"[{name}] Embedding queries"):
        batch = remaining[batch_start : batch_start + EMBED_BATCH_SIZE]
        embs  = wav_to_embedding(batch, processor, model)
        chunk_embeds.append(embs)
        processed += len(batch)

        if processed % SAVE_EVERY < EMBED_BATCH_SIZE or processed == len(query_paths):
            np.save(partial_query_emb, np.concatenate(chunk_embeds, axis=0))
            with open(progress_path, "w") as f:
                json.dump({"done": processed, "total": len(query_paths)}, f)

    qembs = np.concatenate(chunk_embeds, axis=0).astype(np.float32)
    np.save(final_query_emb, qembs)
    query_emb_cache[name] = qembs
    print(f"[{name}] Query embeddings saved: {qembs.shape} → {final_query_emb}")

    for p in [partial_query_emb, progress_path]:
        if os.path.exists(p):
            os.remove(p)

    del model
    torch.cuda.empty_cache()

print("\nAll query embeddings ready.")

## Section 5 — Recall@1,3,5,10 Evaluation

Searches each model's FAISS index with the query embeddings, then computes Recall@K.
Results are printed as a comparison table and saved to Drive.

In [ ]:
# ── Section 5A: FAISS search ──────────────────────────────────────────────────
# Retrieve top-max_K gallery items for every query
max_k   = max(RECALL_AT_K)
all_I   = {}  # name → (N_q, max_k) int array of gallery indices
all_D   = {}  # name → (N_q, max_k) float array of similarity scores

for cfg in CHECKPOINT_CONFIGS:
    name = cfg["name"]
    if name not in faiss_indexes or name not in query_emb_cache:
        print(f"[{name}] Skipping — index or query embeddings missing.")
        continue

    index  = faiss_indexes[name]
    qembs  = query_emb_cache[name]  # (N_q, 768) float32

    print(f"[{name}] Searching FAISS ({len(query_paths)} queries, top-{max_k})...")
    D, I = index.search(qembs, max_k)
    all_D[name] = D
    all_I[name] = I
    print(f"[{name}] Search done. Shape: I={I.shape}  D={D.shape}")

print("\nSearch complete for:", list(all_I.keys()))

In [ ]:
# ── Section 5B: Recall@K computation ─────────────────────────────────────────
import matplotlib.pyplot as plt

results_rows = []

for cfg in CHECKPOINT_CONFIGS:
    name = cfg["name"]
    if name not in all_I:
        continue

    recall = recall_at_k(
        query_labels   = query_labels,
        gallery_labels = gallery_labels,
        scores         = all_I[name],
        ks             = RECALL_AT_K,
    )
    row = {"model": name, "retrieval_unit": RETRIEVAL_UNIT}
    row.update({f"R@{k}": round(v * 100, 2) for k, v in recall.items()})
    results_rows.append(row)

    print(f"[{name}]  " +
          "  ".join(f"R@{k}={v*100:.1f}%" for k, v in recall.items()))

results_df = pd.DataFrame(results_rows)
print("\n", results_df.to_string(index=False))

# Save to Drive
results_csv = os.path.join(EVAL_BASE, "results", f"recall_at_k_{RETRIEVAL_UNIT}.csv")
results_df.to_csv(results_csv, index=False)
print(f"\nResults saved → {results_csv}")

In [ ]:
# ── Section 5C: Comparison bar chart ─────────────────────────────────────────
k_cols = [f"R@{k}" for k in RECALL_AT_K if f"R@{k}" in results_df.columns]
n_models = len(results_df)
n_k      = len(k_cols)
x        = np.arange(n_k)
width    = 0.8 / max(n_models, 1)

fig, ax = plt.subplots(figsize=(max(8, n_k * 2), 5))
colors = plt.cm.tab10.colors

for i, row in results_df.iterrows():
    vals = [row[c] for c in k_cols]
    bars = ax.bar(x + i * width, vals, width, label=row["model"],
                  color=colors[i % len(colors)], alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{val:.1f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x + width * (n_models - 1) / 2)
ax.set_xticklabels(k_cols, fontsize=11)
ax.set_ylabel("Recall (%)")
ax.set_title(f"Recall@K by Model  (retrieval unit: {RETRIEVAL_UNIT})")
ax.set_ylim(0, 110)
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()

plot_path = os.path.join(EVAL_BASE, "results", f"recall_at_k_{RETRIEVAL_UNIT}.png")
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"Plot saved → {plot_path}")

In [ ]:
# ── Section 5D: Per-composer Recall@1 breakdown ───────────────────────────────
# Shows which composers are hardest to retrieve — useful for error analysis.

for cfg in CHECKPOINT_CONFIGS:
    name = cfg["name"]
    if name not in all_I:
        continue

    print(f"\n── [{name}] Per-{RETRIEVAL_UNIT} Recall@1 ──")
    I = all_I[name]
    gallery_labels_arr = np.array(gallery_labels)
    unique_labels = sorted(set(query_labels))

    rows = []
    for label in unique_labels:
        idxs  = [i for i, l in enumerate(query_labels) if l == label]
        hits  = sum(
            1 for i in idxs
            if gallery_labels_arr[I[i, 0]] == label
        )
        rows.append({RETRIEVAL_UNIT: label, "queries": len(idxs),
                     "R@1": round(hits / len(idxs) * 100, 1)})

    per_df = pd.DataFrame(rows).sort_values("R@1")
    print(per_df.to_string(index=False))

    per_csv = os.path.join(EVAL_BASE, "results",
                           f"{name}_per_{RETRIEVAL_UNIT}_recall1.csv")
    per_df.to_csv(per_csv, index=False)
    print(f"Saved → {per_csv}")

In [ ]:
# ── Section 5E: Qualitative — top-5 neighbours for a sample query ────────────
# Pick the first query from each composer and show its top-5 gallery matches.

SAMPLE_QUERIES = 3   # how many sample queries to display

for cfg in CHECKPOINT_CONFIGS[:1]:   # just the first model for brevity
    name = cfg["name"]
    if name not in all_I:
        continue
    I = all_I[name]
    D = all_D[name]

    print(f"\n── [{name}] Top-5 neighbours for {SAMPLE_QUERIES} sample queries ──")
    for qi in range(min(SAMPLE_QUERIES, len(query_paths))):
        qpath  = query_paths[qi]
        qlabel = query_labels[qi]
        print(f"\nQuery [{qi}]: {os.path.basename(qpath)}  label={qlabel}")
        for rank, (gi, score) in enumerate(zip(I[qi], D[qi])):
            glabel = gallery_labels[gi]
            gpath  = gallery_paths[gi]
            match  = "✓" if glabel == qlabel else " "
            print(f"  Rank {rank+1} {match}  score={score:.4f}  {glabel:20s}  {os.path.basename(gpath)}")